Preprocessing

In [15]:
!pip install groq

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [16]:
!pip install pandas
!pip install rank_bm25

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [17]:
import pandas as pd
import re

# Load your scraped CSV
df = pd.read_csv("/kaggle/input/extended-pubmed/pubmed_heart_failure_extended.csv")

print("Initial rows:", len(df))

# ---- 1. Remove duplicates ----
df.drop_duplicates(subset=["title", "abstract"], inplace=True)

# ---- 2. Remove empty/invalid abstracts ----
df = df[df['abstract'].notna() & (df['abstract'].str.strip() != "")]

# ---- 3. Remove very short abstracts (noise) ----
df = df[df['abstract'].str.len() > 200]

# ---- 4. Clean text: remove excessive whitespace, unicode issues ----
def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    text = text.replace("\n", " ")
    return text.strip()

df['title'] = df['title'].apply(clean_text)
df['abstract'] = df['abstract'].apply(clean_text)

print("Processed rows:", len(df))

# ---- 5. Save cleaned dataset ----
df.to_csv("pubmed_cleaned.csv", index=False)

print("Saved cleaned CSV as pubmed_cleaned.csv")


Initial rows: 1500
Processed rows: 1488
Saved cleaned CSV as pubmed_cleaned.csv


Chunking

In [18]:
import pandas as pd
import re
import uuid
import nltk
from nltk.tokenize import sent_tokenize

# --- 1. Setup NLTK ---
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

# --- 2. Load & Fix Data ---
df = pd.read_csv("/kaggle/input/extended-pubmed/pubmed_heart_failure_extended.csv")

# Helper functions to extract metadata
def extract_pmid(url_str):
    """Extracts numbers from the end of a PubMed URL."""
    try:
        # Looks for the last group of digits in the URL
        match = re.search(r'(\d+)/?$', str(url_str))
        return match.group(1) if match else "0"
    except:
        return "0"

def extract_year(date_str):
    """Extracts the first 4-digit year found in the string."""
    try:
        # Looks for 4 consecutive digits (e.g., 2015)
        match = re.search(r'(\d{4})', str(date_str))
        return match.group(1) if match else "0"
    except:
        return "0"

# Apply extractions to create the columns your pipeline expects
print("Extracting PMIDs and Years...")
df['pmid'] = df['url'].apply(extract_pmid)
df['year'] = df['pub_date'].apply(extract_year)

# Clean Text
def clean_text(text):
    text = re.sub(r"\s+", " ", str(text))
    return text.strip()

df['title'] = df['title'].apply(clean_text)
df['abstract'] = df['abstract'].apply(clean_text)

# Filter garbage
df = df[df['abstract'].str.len() > 50]  # Remove empty abstracts

print(f"Data ready. Rows: {len(df)}")
print(f"Sample Year: {df['year'].iloc[0]} | Sample PMID: {df['pmid'].iloc[0]}")

# --- 3. Chunking with Metadata ---
def chunk_text(text, max_tokens=350, overlap=100):
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = []
    current_word_count = 0
    
    for sentence in sentences:
        sentence_words = sentence.split()
        sentence_len = len(sentence_words)
        
        if current_word_count + sentence_len > max_tokens:
            chunks.append(" ".join(current_chunk))
            if overlap > 0 and current_chunk:
                all_words = " ".join(current_chunk).split()
                overlap_words = all_words[-overlap:]
                current_chunk = [" ".join(overlap_words)]
                current_word_count = len(overlap_words)
            else:
                current_chunk = []
                current_word_count = 0
        
        current_chunk.append(sentence)
        current_word_count += sentence_len
    
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks

chunk_rows = []
print("Chunking text...")

for idx, row in df.iterrows():
    # Now we grab the columns we just created
    title = row.get("title", "")
    abstract = row.get("abstract", "")
    pmid = row.get("pmid", "0")
    year = row.get("year", "0")
    
    if len(abstract) < 10: continue

    text = f"{title}. {abstract}"
    chunks = chunk_text(text)

    for chunk in chunks:
        chunk_rows.append({
            "chunk_id": str(uuid.uuid4()),
            "pmid": pmid,   # This will now have the ID from the URL
            "year": year,   # This will now have the year like "2015"
            "title": title,
            "text_chunk": chunk
        })

# Save
chunk_df = pd.DataFrame(chunk_rows)
chunk_df.to_csv("pubmed_chunks.csv", index=False)
print(f"Saved pubmed_chunks.csv with {len(chunk_df)} chunks.")

Extracting PMIDs and Years...
Data ready. Rows: 1499
Sample Year: 2007 | Sample PMID: 17699180
Chunking text...
Saved pubmed_chunks.csv with 1515 chunks.


In [19]:
!pip install sentence-transformers faiss-cpu pandas tqdm


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [20]:
import pandas as pd
import faiss
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

# Load chunked dataset
df = pd.read_csv("pubmed_chunks.csv")

# Choose a medical-domain embedding model
model_name = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"  # best for biomedical text
model = SentenceTransformer(model_name)

# Convert text column to list
texts = df["text_chunk"].tolist()

# Generate embeddings
print("Generating embeddings...")
embeddings = model.encode(texts, batch_size=16, show_progress_bar=True)

# Convert embeddings to float32 (required by FAISS)
embeddings = np.array(embeddings).astype("float32")

# Build FAISS index
d = embeddings.shape[1]  # vector dimension
index = faiss.IndexFlatL2(d)

print("Building FAISS index...")
index.add(embeddings)

# Save FAISS index
faiss.write_index(index, "pubmed_faiss.index")

# Save metadata separately
df.to_csv("pubmed_metadata.csv", index=False)

print("FAISS index saved as pubmed_faiss.index")
print("Metadata saved as pubmed_metadata.csv")
print("Total vectors stored:", index.ntotal)


Generating embeddings...


Batches:   0%|          | 0/95 [00:00<?, ?it/s]

Building FAISS index...
FAISS index saved as pubmed_faiss.index
Metadata saved as pubmed_metadata.csv
Total vectors stored: 1515


In [21]:
# --- Missing Helper Functions ---

def faiss_search(query: str, top_k: int = 50):
    """
    Encodes the query and searches the FAISS index.
    Requires 'model' and 'index' to be loaded globally.
    """
    # 1. Embed query to vector
    qvec = model.encode([query]).astype("float32")
    
    # 2. Search index
    distances, indices = index.search(qvec, top_k)
    
    # 3. Return 1D arrays (first batch element)
    return distances[0], indices[0]

def dist_to_similarity(dist):
    """
    Converts Euclidean (L2) distance to a similarity score (0 to 1).
    1.0 = identical, 0.0 = infinite distance.
    """
    return 1.0 / (1.0 + dist)

In [22]:
import numpy as np
import datetime

# --- 1. The Math: Sigmoidal Decay ---
def sigmoid_decay_weight(pub_year, threshold_age=10, steepness=0.5):
    """
    Sigmoid Function (S-Curve):
    - Papers newer than 'threshold_age' keep Score ~ 1.0 (The "Shelf").
    - Papers older than threshold drop off rapidly (The "Cliff").
    """
    try:
        current_year = datetime.datetime.now().year
        age = max(0, current_year - int(pub_year))
        
        # The equation: 1 / (1 + e^(k * (age - threshold)))
        # We add a tiny epsilon to avoiding potential overflow issues, though rare here.
        weight = 1 / (1 + np.exp(steepness * (age - threshold_age)))
        return float(weight)
    except:
        return 0.5 # Neutral weight for unknown years

# --- 2. The Reranker: Apply Sigmoid ---
def rerank_with_sigmoid(distances, indices, threshold=10, steepness=0.5, top_n=6):
    items = []
    for dist, idx in zip(distances, indices):
        try:
            row = metadata.iloc[idx]
        except IndexError:
            continue
            
        # Get Similarity
        sim = dist_to_similarity(dist)
        
        # Get Sigmoid Weight
        decay_w = sigmoid_decay_weight(row.get("year", None), threshold, steepness)
        
        # Final Score
        final_score = sim * decay_w
        
        items.append({
            "pmid": row.get("pmid", ""),
            "year": row.get("year", ""),
            "title": row.get("title", "")[:200],
            "chunk": row.get("text_chunk", "")[:1000],
            "similarity": sim,
            "decay_weight": decay_w,
            "final_score": final_score
        })
        
    # Sort by Final Score
    return sorted(items, key=lambda x: x["final_score"], reverse=True)[:top_n]

temporal decay function

In [23]:
import numpy as np
import datetime

def temporal_decay(pub_year, lambda_decay=0.05):
    """
    pub_year: publication year of the chunk
    lambda_decay: decay rate (0.01 = slow decay, 0.1 = fast decay)
    """
    current_year = datetime.datetime.now().year
    age = current_year - int(pub_year) if not np.isnan(pub_year) else 0

    return np.exp(-lambda_decay * age)


faiss search + temporal reranking

In [24]:
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

# Load embedding model, metadata, and FAISS index
model = SentenceTransformer("cambridgeltl/SapBERT-from-PubMedBERT-fulltext")
metadata = pd.read_csv("pubmed_metadata.csv")
index = faiss.read_index("pubmed_faiss.index")

def search_with_temporal_decay(query, top_k=10, lambda_decay=0.05):
    """
    Perform FAISS search → get similarity → apply temporal decay → rerank results.
    """

    # Step 1: Embed query
    query_vec = model.encode([query]).astype("float32")

    # Step 2: Raw FAISS search
    distances, indices = index.search(query_vec, top_k)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        row = metadata.iloc[idx]

        pub_year = row.get("year", None)
        decay_w = temporal_decay(pub_year, lambda_decay)

        # Convert FAISS L2 distance to similarity
        raw_similarity = 1 / (1 + dist)

        # Apply decay
        final_score = raw_similarity * decay_w

        results.append({
            "chunk_id": row["chunk_id"],
            "pmid": row["pmid"],
            "year": pub_year,
            "title": row["title"],
            "chunk": row["text_chunk"],
            "raw_similarity": float(raw_similarity),
            "decay_weight": float(decay_w),
            "final_score": float(final_score)
        })

    # Step 3: Re-rank by temporal-decayed score
    results = sorted(results, key=lambda x: x["final_score"], reverse=True)
    return results


testing temporal decay RAG

In [25]:
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
import numpy as np

# 1. Setup Cross-Encoder (The "Judge")
# MS MARCO is excellent for determining if a document ACTUALLY answers a query
print("Loading Cross-Encoder...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# 2. Setup BM25 (The "Keyword Hunter")
# We need to tokenize the text chunks for BM25
print("Building BM25 Index...")
corpus = metadata['text_chunk'].tolist()
tokenized_corpus = [doc.lower().split(" ") for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

print("Advanced RAG components ready.")

Loading Cross-Encoder...
Building BM25 Index...
Advanced RAG components ready.


In [26]:
def search_hybrid(query, top_k=50):
    """
    Combines FAISS (Vector) and BM25 (Keyword) results.
    """
    # 1. Vector Search
    vec_dists, vec_idxs = faiss_search(query, top_k=top_k)
    
    # 2. Keyword Search
    tokenized_query = query.lower().split(" ")
    # Get top scores from BM25
    bm25_scores = bm25.get_scores(tokenized_query)
    # Get top N indices
    bm25_idxs = np.argsort(bm25_scores)[::-1][:top_k]
    
    # 3. Reciprocal Rank Fusion (RRF)
    # Score = 1 / (rank + 60)
    doc_scores = {}
    
    # Process Vector Results
    for rank, idx in enumerate(vec_idxs):
        if idx == -1: continue
        doc_scores[idx] = doc_scores.get(idx, 0) + (1 / (rank + 60))
        
    # Process BM25 Results
    for rank, idx in enumerate(bm25_idxs):
        doc_scores[idx] = doc_scores.get(idx, 0) + (1 / (rank + 60))
    
    # Sort by fused score and return top indices
    sorted_idxs = sorted(doc_scores.keys(), key=lambda x: doc_scores[x], reverse=True)[:top_k]
    return sorted_idxs

def get_dynamic_lambda(query):
    """
    Decides how much 'Time Decay' to apply based on the user's intent.
    """
    q_lower = query.lower()
    
    # Case A: User wants history/mechanisms -> Decay is bad (Keep old papers)
    if any(x in q_lower for x in ['history', 'mechanism', 'pathology', 'origin', 'classic']):
        return 0.005 # Extremely low decay
        
    # Case B: User wants current standards -> Decay is good (Punish old papers)
    if any(x in q_lower for x in ['latest', 'current', 'guideline', 'recommendation', '2023', '2024', 'update']):
        return 0.15 # Strong decay
        
    # Default
    return 0.05

def rerank_advanced(query, indices, lambda_decay):
    """
    Takes candidate indices -> Cross-Encoder Score -> Time Decay -> Final List
    """
    # 1. Prepare pairs for Cross-Encoder
    candidates = []
    valid_indices = []
    
    for idx in indices:
        try:
            row = metadata.iloc[idx]
            candidates.append([query, row['text_chunk']])
            valid_indices.append(idx)
        except:
            continue
            
    # 2. Get Semantic Relevance Scores (Logits)
    ce_scores = cross_encoder.predict(candidates)
    
    # 3. Normalize scores (Logit -> 0..1) using Sigmoid
    ce_scores_norm = 1 / (1 + np.exp(-ce_scores))
    
    # 4. Apply Time Decay to the Semantic Score
    final_results = []
    for i, idx in enumerate(valid_indices):
        row = metadata.iloc[idx]
        
        # Calculate Time Weight
        decay_w = temporal_decay_weight(row.get('year', np.nan), lambda_decay)
        
        # Combine: Relevance * Freshness
        relevance = ce_scores_norm[i]
        final_score = relevance * decay_w
        
        final_results.append({
            "pmid": row.get('pmid', '0'),
            "year": row.get('year', '0'),
            "title": row.get('title', ''),
            "chunk": row.get('text_chunk', ''),
            "relevance": float(relevance),
            "decay": float(decay_w),
            "final_score": float(final_score)
        })
        
    # 5. Sort by final score
    return sorted(final_results, key=lambda x: x['final_score'], reverse=True)

In [27]:
import numpy as np
import datetime

# --- 1. Math Helper Functions ---

def temporal_decay_weight(pub_year, lambda_decay=0.05):
    """
    E-TVD: Exponentially punishes older papers.
    """
    try:
        current_year = datetime.datetime.now().year
        year = int(pub_year)
        age = max(0, current_year - year)
        return float(np.exp(-lambda_decay * age))
    except:
        return 0.5 # Default if year is missing

def sigmoid_decay_weight(pub_year, threshold=10, steepness=0.5):
    """
    Sigmoid: Keeps scores high until 'threshold' years, then drops (The "Cliff").
    """
    try:
        current_year = datetime.datetime.now().year
        year = int(pub_year)
        age = max(0, current_year - year)
        return float(1 / (1 + np.exp(steepness * (age - threshold))))
    except:
        return 0.5

def normalized_recency(pub_year, min_year=1980, max_year=None):
    """
    Normalizes year to [0,1] range for BioScore.
    """
    try:
        y = int(pub_year)
        if max_year is None: max_year = datetime.datetime.now().year
        y = np.clip(y, min_year, max_year)
        return float((y - min_year) / (max_year - min_year))
    except:
        return 0.0

def compute_bioscore(similarity, pub_year, alpha=0.7, beta=0.3):
    """
    BioScore: Linear combination of Similarity + Recency.
    """
    rec = normalized_recency(pub_year)
    return float(similarity * alpha + rec * beta)

def dist_to_similarity(dist):
    """
    Converts FAISS L2 distance to Similarity (0..1).
    """
    return float(1.0 / (1.0 + dist))

# --- 2. The Reranking Functions (Missing Part) ---

def rerank_with_etvd(distances, indices, lambda_decay=0.05, top_n=6):
    items = []
    for dist, idx in zip(distances, indices):
        if idx == -1: continue
        try: row = metadata.iloc[idx]
        except: continue
            
        sim = dist_to_similarity(dist)
        decay_w = temporal_decay_weight(row.get("year", None), lambda_decay)
        final_score = sim * decay_w
        
        items.append({
            "pmid": row.get("pmid", "0"),
            "year": row.get("year", "0"),
            "title": row.get("title", "")[:200],
            "chunk": row.get("text_chunk", "")[:1000],
            "similarity": sim,
            "decay_weight": decay_w,
            "final_score": final_score
        })
    # Sort by final score
    return sorted(items, key=lambda x: x["final_score"], reverse=True)[:top_n]

def rerank_with_sigmoid(distances, indices, threshold=10, steepness=0.5, top_n=6):
    items = []
    for dist, idx in zip(distances, indices):
        if idx == -1: continue
        try: row = metadata.iloc[idx]
        except: continue
            
        sim = dist_to_similarity(dist)
        decay_w = sigmoid_decay_weight(row.get("year", None), threshold, steepness)
        final_score = sim * decay_w
        
        items.append({
            "pmid": row.get("pmid", "0"),
            "year": row.get("year", "0"),
            "title": row.get("title", "")[:200],
            "chunk": row.get("text_chunk", "")[:1000],
            "similarity": sim,
            "decay_weight": decay_w,
            "final_score": final_score
        })
    return sorted(items, key=lambda x: x["final_score"], reverse=True)[:top_n]

def rerank_with_bioscore(distances, indices, alpha=0.7, beta=0.3, top_n=6):
    items = []
    for dist, idx in zip(distances, indices):
        if idx == -1: continue
        try: row = metadata.iloc[idx]
        except: continue
            
        sim = dist_to_similarity(dist)
        bs = compute_bioscore(sim, row.get("year", None), alpha, beta)
        
        items.append({
            "pmid": row.get("pmid", "0"),
            "year": row.get("year", "0"),
            "title": row.get("title", "")[:200],
            "chunk": row.get("text_chunk", "")[:1000],
            "similarity": sim,
            "bioscore": bs
        })
    return sorted(items, key=lambda x: x["bioscore"], reverse=True)[:top_n]

In [28]:
def answer_with_rag(query: str,
                    method: str = "etvd", # Options: "etvd", "bioscore", "sigmoid"
                    top_k: int = 50,
                    context_k: int = 6,
                    # E-TVD Params
                    lambda_decay: float = 0.05,
                    # Sigmoid Params
                    sig_threshold: int = 10,
                    sig_steepness: float = 0.5,
                    # BioScore Params
                    alpha: float = 0.7, beta: float = 0.3,
                    use_openai: bool = True):
    
    # 1. Retrieve
    dists, idxs = faiss_search(query, top_k=top_k)
    
    # 2. Rerank based on method
    if method == "etvd":
        reranked = rerank_with_etvd(dists, idxs, lambda_decay=lambda_decay, top_n=context_k)
        
    elif method == "sigmoid":
        reranked = rerank_with_sigmoid(dists, idxs, threshold=sig_threshold, steepness=sig_steepness, top_n=context_k)
        
    else: # bioscore
        reranked = rerank_with_bioscore(dists, idxs, alpha=alpha, beta=beta, top_n=context_k)

    # 3. Build Context
    is_using_openai = use_openai and OPENAI_AVAILABLE and os.getenv("OPENAI_API_KEY")
    is_using_groq = os.environ.get("GROQ_API_KEY", "").startswith("gsk_")
    max_ctx_chars = 3000 if (is_using_openai or is_using_groq) else 1000
        
    context = build_context(reranked, max_chars=max_ctx_chars)

    # 4. Generate Answer
    system_prompt = """You are a clinical assistant. Answer based ONLY on the context. Cite sources like [1]."""
    user_prompt = f"QUESTION: {query}\n\nCONTEXT:\n{context}\n\nANSWER:"
    
    if is_using_openai or is_using_groq:
        if is_using_groq:
            answer = generate_groq(system_prompt, user_prompt)
        else:
            answer = generate_openai_chat(system_prompt, user_prompt)
    else:
        answer = generate_local(f"question: {query} context: {context}"[:1000])

    return {
        "query": query,
        "method": method,
        "reranked": reranked,
        "answer": answer
    }

In [29]:
# Query: "Standard of care for heart failure"
# (This typically benefits from recent papers, so Sigmoid might perform well)
query = "standard of care for heart failure treatment"

print(f"QUERY: {query}")

# --- Run E-TVD (Exponential) ---
print("\n" + "="*40)
print("METHOD 1: E-TVD (Exponential Decay)")
print("Logic: Punishes age immediately. 1 year old is worse than 0 years old.")
res_etvd = answer_with_rag(query, method="etvd", lambda_decay=0.05)
print(f"ANSWER: {res_etvd['answer']}\n")
print("Top 3 Sources:")
for r in res_etvd['reranked'][:3]:
    print(f" - Year: {r['year']} | Weight: {r['decay_weight']:.2f} | Title: {r['title'][:80]}...")

# --- Run Sigmoid (Cliff Decay) ---
print("\n" + "="*40)
print("METHOD 2: Sigmoid (The 'Guideline' Curve)")
print("Logic: Keeps papers high score until 10 years old, then drops them.")
res_sig = answer_with_rag(query, method="sigmoid", sig_threshold=10, sig_steepness=0.5)
print(f"ANSWER: {res_sig['answer']}\n")
print("Top 3 Sources:")
for r in res_sig['reranked'][:3]:
    print(f" - Year: {r['year']} | Weight: {r['decay_weight']:.2f} | Title: {r['title'][:80]}...")

QUERY: standard of care for heart failure treatment

METHOD 1: E-TVD (Exponential Decay)
Logic: Punishes age immediately. 1 year old is worse than 0 years old.


NameError: name 'OPENAI_AVAILABLE' is not defined

In [ ]:
# Query: "Standard of care for heart failure"
# (This typically benefits from recent papers, so Sigmoid might perform well)
query = "standard of care for heart failure treatment"

print(f"QUERY: {query}")

# --- Run E-TVD (Exponential) ---
print("\n" + "="*40)
print("METHOD 1: E-TVD (Exponential Decay)")
print("Logic: Punishes age immediately. 1 year old is worse than 0 years old.")
res_etvd = answer_with_rag(query, method="etvd", lambda_decay=0.05)
print(f"ANSWER: {res_etvd['answer']}\n")
print("Top 3 Sources:")
for r in res_etvd['reranked'][:3]:
    print(f" - Year: {r['year']} | Weight: {r['decay_weight']:.2f} | Title: {r['title'][:80]}...")

# --- Run Sigmoid (Cliff Decay) ---
print("\n" + "="*40)
print("METHOD 2: Sigmoid (The 'Guideline' Curve)")
print("Logic: Keeps papers high score until 10 years old, then drops them.")
res_sig = answer_with_rag(query, method="sigmoid", sig_threshold=10, sig_steepness=0.5)
print(f"ANSWER: {res_sig['answer']}\n")
print("Top 3 Sources:")
for r in res_sig['reranked'][:3]:
    print(f" - Year: {r['year']} | Weight: {r['decay_weight']:.2f} | Title: {r['title'][:80]}...")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- 1. Define Missing Helpers (The Fix) ---
# These are required because 'faiss_search' was missing from your previous cell
def dist_to_similarity(dist):
    return 1.0 / (1.0 + dist)

def faiss_search(query: str, top_k: int = 50):
    qvec = model.encode([query]).astype("float32")
    distances, indices = index.search(qvec, top_k)
    return distances[0], indices[0]

# --- 2. Generate Data ---
query = "beta blockers heart failure"

# Step A: Get raw FAISS results
dists, idxs = faiss_search(query, top_k=50)

# Step B: Get Results for BOTH Methods (for comparison)
# Method 1: E-TVD (Exponential)
results_etvd = rerank_with_etvd(dists, idxs, lambda_decay=0.05, top_n=50)

# Method 2: Sigmoid (Guideline Cliff)
# We assume 'rerank_with_sigmoid' is defined from your previous steps
results_sig = rerank_with_sigmoid(dists, idxs, threshold=10, steepness=0.5, top_n=50)

# --- 3. Visualization: Side-by-Side Comparison ---
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Plot 1: E-TVD
sims_1 = [d["similarity"] for d in results_etvd]
scores_1 = [d["final_score"] for d in results_etvd]
weights_1 = [d["decay_weight"] for d in results_etvd]

sc1 = axes[0].scatter(sims_1, scores_1, c=weights_1, cmap='viridis', s=100, alpha=0.8)
axes[0].set_title("Method A: E-TVD (Exponential Decay)\nPunishes age immediately")
axes[0].set_xlabel("Semantic Similarity")
axes[0].set_ylabel("Final Score")
axes[0].grid(True, alpha=0.3)
plt.colorbar(sc1, ax=axes[0], label="Decay Weight")

# Plot 2: Sigmoid
sims_2 = [d["similarity"] for d in results_sig]
scores_2 = [d["final_score"] for d in results_sig]
weights_2 = [d["decay_weight"] for d in results_sig]

sc2 = axes[1].scatter(sims_2, scores_2, c=weights_2, cmap='plasma', s=100, alpha=0.8)
axes[1].set_title("Method B: Sigmoidal Decay\nKeeps recent papers high, drops old ones")
axes[1].set_xlabel("Semantic Similarity")
axes[1].set_ylabel("Final Score")
axes[1].grid(True, alpha=0.3)
plt.colorbar(sc2, ax=axes[1], label="Decay Weight")

plt.tight_layout()
plt.show()

In [ ]:
"""
Full RAG Pipeline with Multiple Ranking Strategies
- Loads FAISS index and metadata
- Ranking methods:
    * Ebbinghaus Temporal Vector Decay (E-TVD) [Exponential punishment]
    * Sigmoidal Decay [Guideline-cliff model]
    * BioScore baseline [Linear combination]
- Builds LLM context from top chunks
- Calls LLM to generate citation-backed answer
"""

import os
import datetime
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Any
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()


# --- 2. SETUP API KEYS (Critical Fix) ---
try:
    # 1. Fetch from Kaggle Secrets
    groq_key = user_secrets.get_secret("GROQ_API_KEY")
    
    # 2. ASSIGN TO ENVIRONMENT (This was missing!)
    # The Groq client and your functions look for this specific Env Var
    if groq_key:
        os.environ["GROQ_API_KEY"] = groq_key
        print("Groq API Key successfully loaded into environment.")
except Exception as e:
    print(f"Groq Key not found in secrets: {e}")

# Now this check will accurately reflect the Environment state
is_using_groq = os.environ.get("GROQ_API_KEY", "").startswith("gsk_")

if is_using_groq:
    print("System will use Groq (Llama-3) instead of OpenAI.")
else:
    print("WARNING: Groq Key not found. System will fallback to Local or crash.")

# ----- Config -----
FAISS_INDEX_PATH = "pubmed_faiss.index"
METADATA_PATH = "pubmed_metadata.csv"   
EMBEDDING_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext" 
TOP_K = 50            # FAISS raw retrieval size
CONTEXT_K = 6         # number of final context chunks passed to LLM
LAMBDA_DECAY = 0.05   # E-TVD lambda
SIG_THRESHOLD = 10    # Sigmoid: years before decay starts (The "Shelf")
SIG_STEEPNESS = 0.5   # Sigmoid: how fast it drops after threshold (The "Cliff")
ALPHA = 0.7           # BioScore weight for similarity
BETA = 0.3            # BioScore weight for recency
MAX_TOKENS_CONTEXT = 2500 
OPENAI_MODEL = "gpt-4o-mini"
LOCAL_MODEL_NAME = "google/flan-t5-large"

# ----- Helpers -----

print("Loading metadata and FAISS index...")
# Ensure these files exist in your environment
if os.path.exists(METADATA_PATH) and os.path.exists(FAISS_INDEX_PATH):
    metadata = pd.read_csv(METADATA_PATH)
    index = faiss.read_index(FAISS_INDEX_PATH)
    embed_model = SentenceTransformer(EMBEDDING_MODEL)
    # Validation
    # assert "text_chunk" in metadata.columns, "metadata must contain text_chunk field"
else:
    print("Warning: Index or Metadata not found. Please run ingestion first.")

# --- 1. Math Functions ---

def temporal_decay_weight(pub_year: Any, lambda_decay: float = LAMBDA_DECAY) -> float:
    """ E-TVD weight: exp(-lambda * age) """
    try:
        current_year = datetime.datetime.now().year
        year = int(pub_year)
        age = max(0, current_year - year)
    except Exception:
        return 0.5 # Default for unknown years
    return float(np.exp(-lambda_decay * age))

def sigmoid_decay_weight(pub_year: Any, threshold: int = SIG_THRESHOLD, steepness: float = SIG_STEEPNESS) -> float:
    """ 
    Sigmoid weight: 1 / (1 + e^(k * (age - threshold))) 
    Keeps scores high until 'threshold' years, then drops.
    """
    try:
        current_year = datetime.datetime.now().year
        year = int(pub_year)
        age = max(0, current_year - year)
        # Calculate sigmoid
        weight = 1 / (1 + np.exp(steepness * (age - threshold)))
        return float(weight)
    except Exception:
        return 0.5

def normalized_recency(pub_year: Any, min_year=1980, max_year=None) -> float:
    """ Normalize year to [0,1] """
    try:
        y = int(pub_year)
    except Exception:
        return 0.0
    if max_year is None:
        max_year = datetime.datetime.now().year
    y = np.clip(y, min_year, max_year)
    return float((y - min_year) / (max_year - min_year))

# --- 2. Search & Rerankers ---

def faiss_search(query: str, top_k: int = TOP_K):
    qvec = embed_model.encode([query]).astype("float32")
    distances, indices = index.search(qvec, top_k)
    return distances[0], indices[0]

def dist_to_similarity(dist):
    return float(1.0 / (1.0 + dist))

def compute_bioscore(similarity: float, pub_year: Any, alpha=ALPHA, beta=BETA):
    rec = normalized_recency(pub_year)
    return float(similarity * alpha + rec * beta)

def rerank_with_etvd(distances, indices, lambda_decay=LAMBDA_DECAY, top_n=CONTEXT_K):
    items = []
    for dist, idx in zip(distances, indices):
        if idx == -1: continue
        try: row = metadata.iloc[idx]
        except: continue
            
        sim = dist_to_similarity(dist)
        decay_w = temporal_decay_weight(row.get("year", None), lambda_decay)
        final_score = sim * decay_w
        
        items.append({
            "idx": int(idx),
            "pmid": row.get("pmid", ""),
            "year": row.get("year", ""),
            "title": row.get("title", "")[:200],
            "chunk": row.get("text_chunk", "")[:1000],
            "similarity": sim,
            "decay_weight": decay_w,
            "final_score": final_score
        })
    return sorted(items, key=lambda x: x["final_score"], reverse=True)[:top_n]

def rerank_with_sigmoid(distances, indices, threshold=SIG_THRESHOLD, steepness=SIG_STEEPNESS, top_n=CONTEXT_K):
    items = []
    for dist, idx in zip(distances, indices):
        if idx == -1: continue
        try: row = metadata.iloc[idx]
        except: continue
            
        sim = dist_to_similarity(dist)
        # Use Sigmoid Weight here
        decay_w = sigmoid_decay_weight(row.get("year", None), threshold, steepness)
        final_score = sim * decay_w
        
        items.append({
            "idx": int(idx),
            "pmid": row.get("pmid", ""),
            "year": row.get("year", ""),
            "title": row.get("title", "")[:200],
            "chunk": row.get("text_chunk", "")[:1000],
            "similarity": sim,
            "decay_weight": decay_w,
            "final_score": final_score
        })
    return sorted(items, key=lambda x: x["final_score"], reverse=True)[:top_n]

def rerank_with_bioscore(distances, indices, alpha=ALPHA, beta=BETA, top_n=CONTEXT_K):
    items = []
    for dist, idx in zip(distances, indices):
        if idx == -1: continue
        try: row = metadata.iloc[idx]
        except: continue
            
        sim = dist_to_similarity(dist)
        bs = compute_bioscore(sim, row.get("year", None), alpha, beta)
        
        items.append({
            "idx": int(idx),
            "pmid": row.get("pmid", ""),
            "year": row.get("year", ""),
            "title": row.get("title", "")[:200],
            "chunk": row.get("text_chunk", "")[:1000],
            "similarity": sim,
            "bioscore": bs
        })
    return sorted(items, key=lambda x: x["bioscore"], reverse=True)[:top_n]

# --- 3. Context & Gen ---

def build_context(chunks: List[Dict[str,Any]], max_chars=MAX_TOKENS_CONTEXT*4):
    ctx_parts = []
    total_len = 0
    for i, ch in enumerate(chunks):
        citation = f"[{i+1}] PMID:{ch.get('pmid','?')} | {ch.get('year','?')} | {ch.get('title','')}"
        text = ch.get("chunk","")
        part = f"{citation}\n{ text }\n"
        if total_len + len(part) > max_chars: break
        ctx_parts.append(part)
        total_len += len(part)
    return "\n\n".join(ctx_parts)

def generate_openai_chat(system_prompt, user_prompt, model=OPENAI_MODEL):
    if not OPENAI_AVAILABLE: return "OpenAI not available."
    client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role":"system", "content": system_prompt}, {"role":"user", "content": user_prompt}],
        temperature=0.0
    )
    return resp.choices[0].message.content

# GROQ Wrapper (If you are using Groq)
def generate_groq(system_prompt, user_prompt):
    # Ensure you have 'from groq import Groq' if using this
    try:
        from groq import Groq
        client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
        chat_completion = client.chat.completions.create(
            messages=[{"role": "system", "content": system_prompt},{"role": "user", "content": user_prompt}],
            model="llama3-8b-8192",
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        return f"Groq Error: {e}"

_local_gen_pipeline = None
def generate_local(prompt, max_length=512):
    global _local_gen_pipeline
    if _local_gen_pipeline is None and HF_AVAILABLE:
        _local_gen_pipeline = pipeline("text2text-generation", model=LOCAL_MODEL_NAME, device=-1)
    if _local_gen_pipeline:
        return _local_gen_pipeline(prompt, max_length=max_length)[0]["generated_text"]
    return "No local model available."

# ----- Main Pipeline -----

def answer_with_rag(query: str,
                    method: str = "etvd", # "etvd", "sigmoid", "bioscore"
                    top_k: int = TOP_K,
                    context_k: int = CONTEXT_K,
                    # Method Params
                    lambda_decay: float = LAMBDA_DECAY,
                    sig_threshold: int = SIG_THRESHOLD,
                    sig_steepness: float = SIG_STEEPNESS,
                    alpha: float = ALPHA, beta: float = BETA,
                    use_openai: bool = True):
    
    # 1. Retrieve
    dists, idxs = faiss_search(query, top_k=top_k)
    
    # 2. Rerank
    if method == "sigmoid":
        reranked = rerank_with_sigmoid(dists, idxs, threshold=sig_threshold, steepness=sig_steepness, top_n=context_k)
    elif method == "bioscore":
        reranked = rerank_with_bioscore(dists, idxs, alpha=alpha, beta=beta, top_n=context_k)
    else: # Default etvd
        reranked = rerank_with_etvd(dists, idxs, lambda_decay=lambda_decay, top_n=context_k)

    # 3. Build Context
    is_using_groq = os.environ.get("GROQ_API_KEY", "").startswith("gsk_")
    max_chars = 3000 if (use_openai or is_using_groq) else 1000
    context = build_context(reranked, max_chars=max_chars)

    # 4. Generate
    sys_p = "You are a clinical assistant. Answer using ONLY the context. Cite sources like [1]."
    usr_p = f"QUESTION: {query}\n\nCONTEXT:\n{context}\n\nANSWER:"
    
    if use_openai:
        if is_using_groq:
            ans = generate_groq(sys_p, usr_p)
        elif OPENAI_AVAILABLE:
            ans = generate_openai_chat(sys_p, usr_p)
        else:
            ans = generate_local(f"question: {query} context: {context}")
    else:
        ans = generate_local(f"question: {query} context: {context}")

    return {
        "query": query,
        "method": method,
        "reranked": reranked,
        "answer": ans
    }

# ----- Test / Compare -----
if __name__ == "__main__":
    q = "standard of care for heart failure treatment"
    
    print(f"QUERY: {q}")
    
    # Compare E-TVD vs Sigmoid
    print("\n--- Method: E-TVD (Exponential) ---")
    res1 = answer_with_rag(q, method="etvd", lambda_decay=0.05, use_openai=True)
    print(res1['answer'])
    print("Top Docs:", [f"{r['year']} (Sc: {r['final_score']:.3f})" for r in res1['reranked'][:3]])
    
    print("\n--- Method: Sigmoid (Guideline Cliff) ---")
    res2 = answer_with_rag(q, method="sigmoid", sig_threshold=8, sig_steepness=0.5, use_openai=True)
    print(res2['answer'])
    print("Top Docs:", [f"{r['year']} (Sc: {r['final_score']:.3f})" for r in res2['reranked'][:3]])